# Kerr Free Evolution of a Storage Coherent State


This notebook rewrites `examples/kerr_free_evolution.py` as a workflow tutorial for storage self-Kerr. We prepare a coherent cavity state with target $\alpha = 2$ and let it evolve under the static Hamiltonian with no applied drive.

The physical result is nonlinear phase winding: the cavity mean field rotates, but more importantly the Wigner function bends and eventually departs from a Gaussian coherent-state shape. The notebook stores Wigner snapshots directly in coherent-state $\alpha$ coordinates so the $t = 0$ peak stays centered at the prepared displacement.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from cqed_sim import coherent_state
from examples.workflows.kerr_free_evolution import (
    plot_kerr_wigner_snapshots,
    run_kerr_free_evolution,
    times_us_to_seconds,
)


## Physics / model definition


In [ ]:
times_us = np.array([0.0, 25.0, 50.0, 75.0, 100.0, 125.0, 150.0], dtype=float)
wigner_times_us = np.array([0.0, 125.0, 250.0, 375.0], dtype=float)
initial_cavity_state = coherent_state(2.0)


## Pulse / sequence construction


In [ ]:
times_s = times_us_to_seconds(times_us)
wigner_times_s = times_us_to_seconds(wigner_times_us)
print("This protocol uses no external drive pulses: the cavity simply evolves under the static self-Kerr Hamiltonian.")


## Simulation


In [ ]:
result = run_kerr_free_evolution(
    times_s,
    cavity_state=initial_cavity_state,
    parameter_set="phase_evolution",
    n_cav=30,
    wigner_times_s=wigner_times_s,
    wigner_n_points=91,
    wigner_extent=4.6,
    wigner_coordinate="alpha",
)

mean_field = np.asarray([snapshot.cavity_mean for snapshot in result.snapshots], dtype=np.complex128)
phase_rad = np.unwrap(np.angle(mean_field))
photon_number = np.asarray([snapshot.cavity_photon_number for snapshot in result.snapshots], dtype=float)

t0_snapshot = next(snapshot for snapshot in result.snapshots if snapshot.wigner is not None and np.isclose(snapshot.time_us, 0.0))
t0_wigner = t0_snapshot.wigner["w"]
t0_index = np.unravel_index(np.argmax(t0_wigner), t0_wigner.shape)
t0_peak_alpha = (
    float(t0_snapshot.wigner["xvec"][t0_index[1]]),
    float(t0_snapshot.wigner["yvec"][t0_index[0]]),
)

print("Parameter set:", result.parameter_set.name)
print("Kerr coefficient [Hz]:", result.parameter_set.kerr_hz)
print("t = 0 Wigner peak in alpha coordinates:", t0_peak_alpha)


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].plot(times_us, phase_rad, "o-", color="#4C78A8")
axes[0].set_xlabel("Free-evolution time (us)")
axes[0].set_ylabel("Unwrapped phase of <a> (rad)")
axes[0].set_title("Mean-field phase evolution")

axes[1].plot(times_us, photon_number, "o-", color="#54A24B")
axes[1].set_xlabel("Free-evolution time (us)")
axes[1].set_ylabel("Storage photon number")
axes[1].set_title("Photon number is approximately conserved")
plt.tight_layout()
plt.show()

fig = plot_kerr_wigner_snapshots(result, max_cols=4, show_colorbar=True, coordinate="alpha")
for axis, label in zip(
    fig.axes[:4],
    [r"$t = 0$", r"$t = T_K/4$", r"$t = T_K/2$", r"$t = 3T_K/4$"],
    strict=True,
):
    axis.set_title(label)
plt.show()


## Interpretation


The photon number stays nearly constant because this is unitary free evolution, not ringdown. What changes is the phase picked up by each Fock component, so the coherent state shears in phase space.

That shearing direction is controlled by the sign of the Kerr term documented in `physics_and_conventions/physics_conventions_report.tex`. The notebook also makes the phase-space convention explicit: plotting in coherent-state `alpha` coordinates keeps the initial Wigner peak centered near `(2, 0)`, whereas quadrature coordinates would place the same state near $(\sqrt{2}\,\alpha, 0)$ instead.


## Variations / exercises


- Switch `coordinate="quadrature"` in the plotting helper and verify the expected $\sqrt{2}$ rescaling.
- Change the initial coherent-state amplitude to see how larger `n` content accelerates non-Gaussian distortion.
- Add cavity loss with `NoiseSpec(kappa=...)` if you want to compare Kerr distortion and ringdown.
